# LF4 — nhãn hữu dụng cho tác vụ bệnh đọt/crown (đọc classifier bệnh dùng chung)

Đọc dự đoán out-of-fold của classifier bệnh dùng chung (`labels/disease_clf/oof_predictions.csv`),
lọc ảnh view crown, bỏ phiếu correctness ở mức required-view → `labels/votes/lf4_crown.csv`.
Method + chứng minh (thiết kế "chung model, riêng phiếu", dùng chung cho LF2/3/4/5): xem `docs/LF3_Methodology.md`.

## Cấu hình

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RUNNER = 'local'          # 'local' | 'kaggle'
TASK = '4_crown_disease'
CONF_TAU = 0.50           # cong tin cay; calibrate rieng view crown tren gold seed (xem docs/LF3_Methodology.md)
LF_NAME = 'lf4_crown'
SOURCE = 'coconut-tree-disease'

print('task:', TASK)
print('runner:', RUNNER)
print('conf_tau:', CONF_TAU)
print('lf:', LF_NAME)

task: 4_crown_disease
runner: local
conf_tau: 0.5
lf: lf4_crown


## Đường dẫn + module dùng chung (`src/utils/lf_io.ipynb`)

In [2]:
def build_root(runner):
    if runner == 'local':
        candidates = [Path.cwd(), Path.cwd().parent]
        for cand in candidates:
            probe = cand / 'Dataset' / 'Coconut Tree Disease Dataset'
            if probe.exists():
                return cand
        raise SystemExit('Khong thay Dataset/Coconut Tree Disease Dataset — chay notebook trong repo coconut-iqa')
    if runner == 'kaggle':
        root = Path('/kaggle/input/coconut-iqa')
        probe = root / 'Dataset' / 'Coconut Tree Disease Dataset'
        if not probe.exists():
            raise SystemExit('Khong thay /kaggle/input/coconut-iqa/Dataset/Coconut Tree Disease Dataset')
        return root
    raise SystemExit("RUNNER phai la 'local' hoac 'kaggle'")

ROOT = build_root(RUNNER)
OOF_CSV = ROOT / 'labels' / 'disease_clf' / 'oof_predictions.csv'
VOTES_OUT = ROOT / 'labels' / 'votes' / 'lf4_crown.csv'
if not OOF_CSV.exists():
    raise SystemExit('Khong thay ' + str(OOF_CSV) + ' — chay disease_classifier.ipynb truoc')

UTILS = ROOT / 'src' / 'utils' / 'lf_io.ipynb'
if not UTILS.exists():
    raise SystemExit('Khong thay ' + str(UTILS))
get_ipython().run_line_magic('run', str(UTILS))

print('ROOT:', ROOT)
print('OOF_CSV:', OOF_CSV)
print('VOTES_OUT:', VOTES_OUT)

ROOT: /Users/peggy/Documents/Projects/HK2/coconut-iqa
OOF_CSV: /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/disease_clf/oof_predictions.csv
VOTES_OUT: /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/votes/lf4_crown.csv


/Users/peggy/.pyenv/versions/3.10.13/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## 1. Đọc dự đoán out-of-fold (lọc view crown)

In [3]:
oof = pd.read_csv(OOF_CSV)
crown = oof[oof['gt_task'] == TASK].copy()
print('anh oof:', len(oof))
print('anh view crown:', len(crown))

anh oof: 5798
anh view crown: 470


## 2. Correctness mức view + cổng tin cậy + abstain → phiếu LF4

Công thức phiếu $\lambda_4$ (1/0/abstain) và chứng minh: xem `docs/LF3_Methodology.md`.

In [4]:
def lf4_vote(pred_task, conf):
    if pd.isna(conf):
        return np.nan
    if float(conf) < CONF_TAU:
        return np.nan
    if pred_task == TASK:
        return 1
    return 0

votes = []
for r in crown.itertuples():
    votes.append(lf4_vote(r.pred_task, r.conf))
crown['lf4'] = votes

n_abstain = int(crown['lf4'].isna().sum())
print('phieu 1:', int((crown['lf4'] == 1).sum()))
print('phieu 0:', int((crown['lf4'] == 0).sum()))
print('abstain:', n_abstain, '(' + format(n_abstain / len(crown), '.1%') + ')')

phieu 1: 469
phieu 0: 1
abstain: 0 (0.0%)


## 3. Ghi phiếu ra file riêng `labels/votes/lf4_crown.csv`

In [5]:
rows = []
for r in crown.itertuples():
    rows.append(make_vote(
        lf=LF_NAME,
        image_id=r.image_id,
        task=TASK,
        vote=r.lf4,
        confidence=r.conf,
        source=SOURCE,
        path=r.path,
        pred_class=r.pred_class,
        fold=r.fold,
    ))
write_lf_votes(VOTES_OUT, rows, extra_fields=['pred_class', 'fold'])

ghi: /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/votes/lf4_crown.csv | 470 phieu
  4_crown_disease         : 1=469  0=1


PosixPath('/Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/votes/lf4_crown.csv')

Kiểm định LF4 (đối chiếu gold seed, so sánh giữa các LF) nằm ở notebook so sánh riêng, chạy sau khi mọi LF hoàn tất.